# BBC News Podcast Generator

## Project Brief

This mini-project creates a simple podcast generator using BBC News content as the input source. The app will collect recent BBC News headlines and short summaries, send them to the ChatGPT API, and transform them into a short podcast-style news briefing.

The generated script will then be converted into an audio file using a text-to-speech service. The final result will be a small web app where the user can generate and listen to a short news podcast.

## Main Workflow

Manual BBC News text input
        ↓
ChatGPT API creates podcast script
        ↓
Text-to-speech creates audio file
        ↓
Gradio shows script and audio


## Main Technologies

- Python
- OpenAI ChatGPT API
- Text-to-Speech API
- Gradio
- RSS feed processing
- Environment variables using `.env`

## MVP Goal

The first version will use one BBC News, collect a few recent stories, generate a short podcast script, and create an audio file that can be played from the app.

## Expected Output

- A generated podcast script
- An audio file of the podcast
- A simple Gradio interface to run the project

## Install Required Packages

Run this cell if the required packages are not installed yet.

In [ ]:
%pip install -r requirements.txt

In [ ]:
def read_news_input(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        news_text = file.read()
    return news_text


news_content = read_news_input("news_input.txt")

print("News input loaded successfully:")
print(news_content)

In [ ]:
%pip install openai python-dotenv

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

print("Imports loaded successfully")

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully")
else:
    print("API key not found")

In [ ]:
client = OpenAI(api_key=api_key)

def generate_podcast_script(news_text):
    prompt = f"""
    You are a podcast script writer.

    Create a short news podcast script based on the news content below.

    Requirements:
    - Use clear and simple English
    - Do not copy the original text word-for-word
    - Make it sound natural for audio
    - Include a short intro
    - Include the main story
    - Include a short closing sentence
    - Keep it around 1 to 2 minutes long

    News content:
    {news_text}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You create short and clear podcast scripts."},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [ ]:
podcast_script = generate_podcast_script(news_content)

print(podcast_script)

In [ ]:
def generate_audio(script_text, output_file="podcast.mp3"):
    response = client.audio.speech.create(
        model="tts-1",
        voice="alloy",
        input=script_text
    )

    response.stream_to_file(output_file)

    return output_file

In [ ]:
audio_file = generate_audio(podcast_script)

print(f"Audio file created successfully: {audio_file}")

In [ ]:
from IPython.display import Audio

Audio("podcast.mp3")

In [ ]:
%pip install gradio

In [ ]:
import gradio as gr

print("Gradio loaded successfully")

In [ ]:
def create_podcast_app(news_text):
    podcast_script = generate_podcast_script(news_text)
    audio_file = generate_audio(podcast_script)

    return podcast_script, audio_file

In [ ]:
import gradio as gr

interface = gr.Interface(
    fn=create_podcast_app,
    inputs=gr.Textbox(
        label="Paste news content here",
        lines=10,
        placeholder="Paste a news title and summary here..."
    ),
    outputs=[
        gr.Textbox(label="Generated Podcast Script", lines=15),
        gr.Audio(label="Generated Podcast Audio")
    ],
    title="News Podcast Generator",
    description="Paste news content and generate a short podcast script with audio."
)

interface.launch()